# Paper9 自然资源部输入数据检查

在运行 Paper9 前，先核查权威输入数据：带坡度属性的 DLTB 地类图斑，以及能够解析到村/社区级的行政区数据。容器启动脚本设置 `PAPER9_CONFIG` 后，本 Notebook 会从该环境变量读取配置。

In [ ]:
import os
from pathlib import Path

from IPython.display import HTML, display

from paper9_mnr.notebook_utils import project_root

CONFIG = os.environ.get("PAPER9_CONFIG", "configs/real_data_from_authority_slope.yml")
ROOT = project_root()
NOTEBOOK_OUTPUT_DIR = ROOT / "outputs/notebook"
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT={ROOT}")
print(f"CONFIG={CONFIG}")
print(f"Notebook maps will be written to {NOTEBOOK_OUTPUT_DIR}")

## 必需输入数据要求

In [ ]:
from paper9_mnr.notebook_utils import configured_input_profiles

profiles = configured_input_profiles(CONFIG)
display(profiles[["dataset", "exists", "rows", "crs", "geometry_types", "missing_fields", "path"]])

if not profiles["exists"].all():
    missing = profiles.loc[~profiles["exists"], ["dataset", "path"]]
    raise FileNotFoundError(f"Missing required input datasets:\n{missing.to_string(index=False)}")

field_issues = profiles.loc[profiles["missing_fields"].map(bool), ["dataset", "missing_fields"]]
if not field_issues.empty:
    raise ValueError(f"Missing required fields:\n{field_issues.to_string(index=False)}")

## 输入图层地图

In [ ]:
from paper9_mnr.notebook_utils import input_layers_map_html

input_map = NOTEBOOK_OUTPUT_DIR / "input_layers_map.html"
input_map_html = input_layers_map_html(
    CONFIG,
    max_features=1000,
    output_path=input_map,
)
display(HTML(input_map_html))
print(f"Saved: {input_map}")